In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [2]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

d:\New_file\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [20]:
embeddings.embed_query("Hello Siva")

[-0.018294983,
 -0.004555581,
 0.014337232,
 -0.08360458,
 -0.011260147,
 -0.008444601,
 -0.032982487,
 0.0006205651,
 -0.003030826,
 -0.010871596,
 -0.026838714,
 0.0037231264,
 -0.0016319826,
 0.03566554,
 0.09492289,
 0.000602629,
 -0.004801996,
 -0.00028589982,
 0.0067473613,
 -0.0037063244,
 0.008529183,
 0.004939154,
 -0.00055361236,
 -0.03700829,
 -0.014891271,
 0.009465462,
 0.029252846,
 0.01475381,
 0.039241508,
 0.031381357,
 0.012169348,
 -0.014180522,
 -0.014466636,
 0.011999792,
 0.012614321,
 0.001829531,
 0.007537026,
 0.0074648913,
 0.016405087,
 0.004970708,
 0.0072885402,
 0.02022706,
 -0.006778632,
 0.012519605,
 -0.0037381798,
 -0.0067758937,
 0.00683495,
 -0.019523898,
 0.0014207176,
 0.03374709,
 -0.015411564,
 0.019584628,
 -0.014647028,
 -0.16375668,
 -0.011665417,
 -0.0196766,
 0.011999915,
 -0.0073143914,
 0.002382844,
 0.006560883,
 0.008395403,
 -0.0010649561,
 -0.012203312,
 -0.011936663,
 0.013334856,
 -0.025247157,
 0.010212285,
 0.021132968,
 -0.0150095

In [5]:
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
documents = ["washington DC is the capital of USA",
             "Donald Trump is the president of USA",
             "Narendra Modi is the prime minister of India"]

In [7]:
my_query = "who is president of USA"

In [8]:
embedded_docs = embeddings.embed_documents(documents)
embedded_query = embeddings.embed_query(my_query)

In [9]:
cosine_similarities = cosine_similarity([embedded_query], embedded_docs)

In [ ]:
#for best similarity search, cosine similarity value is closely to 1  or Highest value in array

array([[0.67240526, 0.78551628, 0.5981087 ]])

In [11]:
from sklearn.metrics.pairwise import euclidean_distances

In [ ]:
euclidean_distances_similarities = euclidean_distances([embedded_query], embedded_docs)

In [ ]:
#for euclidean distance, Lowest value is the best similarity search, (Low distance)
euclidean_distances_similarities

array([[0.80943773, 0.65495602, 0.89653921]])

In [19]:
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

In [21]:
len(embeddings.embed_query("Hello Siva"))

3072

In [22]:
index = faiss.IndexFlatL2(3072)

In [25]:
embedidng_function = embeddings

In [28]:
#this is my vectorstore
#In this vectorstore, data is being stored inside the index.
#As of now it is in In memory store

vector_store = FAISS(
    embedding_function = embeddings,
    index = index,
    docstore = InMemoryDocstore(),
    index_to_docstore_id={}
)

In [29]:
vector_store.add_texts(["washington DC is the capital of USA",
             "Donald Trump is the president of USA",
             "Narendra Modi is the prime minister of India"])

['9ee895d6-5c31-4883-a923-7ed165e4cddd',
 'add514ce-881c-4b21-893c-a3121a2b38ac',
 'b9d81c68-a511-4376-89d7-09b5440c87f1']

In [30]:
vector_store.index_to_docstore_id

{0: '9ee895d6-5c31-4883-a923-7ed165e4cddd',
 1: 'add514ce-881c-4b21-893c-a3121a2b38ac',
 2: 'b9d81c68-a511-4376-89d7-09b5440c87f1'}

In [31]:
faiss_index_id = 1

In [33]:
docstore_id = vector_store.index_to_docstore_id[faiss_index_id]

In [34]:
docstore_id

'add514ce-881c-4b21-893c-a3121a2b38ac'

In [35]:
vector_store.docstore.search(docstore_id)

Document(id='add514ce-881c-4b21-893c-a3121a2b38ac', metadata={}, page_content='Donald Trump is the president of USA')

In [36]:
vector_store.docstore.search(docstore_id).page_content

'Donald Trump is the president of USA'

In [37]:
vector_store.docstore.search(docstore_id).metadata

{}

In [ ]:
#Fecthing data from VectorDB after insertrd into it.
#K is no.of simialrty searches
vector_store.similarity_search("who is president of USA", k=1)

[Document(id='add514ce-881c-4b21-893c-a3121a2b38ac', metadata={}, page_content='Donald Trump is the president of USA')]

In [39]:
vector_store.similarity_search("who is president of USA", k=2)

[Document(id='add514ce-881c-4b21-893c-a3121a2b38ac', metadata={}, page_content='Donald Trump is the president of USA'),
 Document(id='9ee895d6-5c31-4883-a923-7ed165e4cddd', metadata={}, page_content='washington DC is the capital of USA')]

In [ ]:
#getting index of particular index_id document
vector_store.index.reconstruct(faiss_index_id)

array([-0.01839727,  0.0179448 ,  0.0263014 , ...,  0.00583152,
        0.00986502, -0.00211253], dtype=float32)